In [1]:
import speech_recognition as sr
from pydub import AudioSegment
import os
from math import ceil

In [3]:
# === CONFIGURACIÓN ===
archivo_original = r"D:\Amatech\Transcriptor y evaluador de llamadas\Arc_prueba\DMCC_ext23580_2025_10_15_13;22;11;989.wav"
archivo_convertido = "temp_pcm.wav"

# === CONVERSIÓN A FORMATO COMPATIBLE ===
print("Convirtiendo el audio...")
sound = AudioSegment.from_file(archivo_original)
sound = sound.set_frame_rate(16000)  # mejor para reconocimiento de voz
sound = sound.set_channels(1)        # convertir a mono
sound.export(archivo_convertido, format="wav")

# === INICIALIZAR RECONOCEDOR ===
recognizer = sr.Recognizer()

# === DIVIDIR AUDIO EN FRAGMENTOS DE 60 SEGUNDOS ===
segment_duration = 60 * 1000  # milisegundos
audio = AudioSegment.from_wav(archivo_convertido)
num_segments = ceil(len(audio) / segment_duration)

print(f"Duración total: {len(audio)/60000:.2f} min")
print(f"Dividiendo en {num_segments} fragmentos de 60s...")

transcripcion = ""

for i in range(num_segments):
    inicio = i * segment_duration
    fin = min((i + 1) * segment_duration, len(audio))
    fragmento = audio[inicio:fin]
    fragment_path = f"temp_fragment_{i}.wav"
    fragmento.export(fragment_path, format="wav")

    with sr.AudioFile(fragment_path) as source:
        recognizer.adjust_for_ambient_noise(source, duration=0.3)
        audio_data = recognizer.record(source)

    try:
        texto = recognizer.recognize_google(audio_data, language="es-ES")
        print(f"Fragmento {i+1}/{num_segments}: OK")
        transcripcion += texto + " "
    except sr.UnknownValueError:
        print(f"Fragmento {i+1}/{num_segments}: no se entendió el audio.")
    except sr.RequestError as e:
        print(f"Error de conexión en el fragmento {i+1}: {e}")
        break
    finally:
        os.remove(fragment_path)

# === MOSTRAR Y GUARDAR RESULTADO ===
if transcripcion.strip():
    print("\n--- TRANSCRIPCIÓN COMPLETA ---\n")
    print(transcripcion)
    with open("transcripcion.txt", "w", encoding="utf-8") as f:
        f.write(transcripcion)
    print("\nLa transcripción completa se guardó en 'transcripcion.txt'.")
else:
    print("No se pudo reconocer texto en el audio completo.")

# === LIMPIEZA ===
if os.path.exists(archivo_convertido):
    os.remove(archivo_convertido)

Convirtiendo el audio...
Duración total: 11.52 min
Dividiendo en 12 fragmentos de 60s...
Error de conexión en el fragmento 1: recognition connection failed: [Errno 11001] getaddrinfo failed
No se pudo reconocer texto en el audio completo.
